In [ ]:
#https://github.com/sinanuozdemir/quick-start-guide-to-llms

In [ ]:
pip install --upgrade openai

In [ ]:
pip install --upgrade cohere

In [3]:
import os
from openai import OpenAI
import cohere

In [ ]:
#set the OpenAI and Cohere keys
%env OPENAI_API_KEY=
%env COHERE_API_KEY=

In [5]:
co = cohere.Client(os.getenv('COHERE_API_KEY'))
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY") )

In [6]:
def test_prompt_openai(prompt, suppress=False, model='gpt-3.5-turbo', **kwargs):
    " a simple function to take in a prompt and run it through a given non-chat model "

    chat_completion = openai_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model=model,
        **kwargs
    )
    answer = chat_completion.choices[0].message.content
    if not suppress:
        print(f'PROMPT:\n------\n{prompt}\n------\nRESPONSE\n------\n{answer}')
    else:
        return answer


In [7]:
def test_prompt_cohere(prompt, suppress=False, model='command-xlarge', **kwargs):
    response = co.generate(
        model=model,
        prompt=prompt,
        **kwargs,
      )
    if not suppress:
        print(f'PROMPT:\n------\n{prompt}\n------\nRESPONSE\n------\n{response.generations[0].text}')

## Just ASK

In [ ]:
test_prompt_openai('Translate to Egyptian Arabic.\n\nWhere is the nearest restaurant?')

PROMPT:
------
Translate to Egyptian Arabic.

Where is the nearest restaurant?
------
RESPONSE
------
فين أقرب مطعم؟


In [ ]:
# depending on the capability of the model, you may need to coax it to structure the output better
# Not Egyptian Arabic
test_prompt_cohere('Translate to Egyptian Arabic.\n\nEnglish: Where is the nearest restaurant?\nEgyptian Arabic:')

PROMPT:
------
Translate to Egyptian Arabic.

English: Where is the nearest restaurant?
Egyptian Arabic:
------
RESPONSE
------
 Egyptian Arabic: "انها ايجارية احضر معينة؟"

This literally translates to: "There is a restaurant, close to here, what?" 
The general format for asking directions in Egyptian Arabic is to declare the object first (in this case, the restaurant), and then ask whether that object is near or far. 
The response would depend on the situation, but generally, the respondent would guide the person asking the question towards a direction, or provide instructions for how to get there. 


# Few-shot learning

Using examples to "teach" GPT-3.5 Trubo what to do


In [ ]:
examples = [
    ('Review: This movie sucks\nSubjective: Yes'),
    ('Review: This tv show was about the ocean\nSubjective: No'),
    ('Review: This book had a lot of flaws\nSubjective: Yes'),

    ('Review: The book was not amazing\nSubjective:'),
]

test_prompt_openai('\n###\n'.join(examples))  # ### is a common few-shot separator

PROMPT:
------
Review: This movie sucks
Subjective: Yes
###
Review: This tv show was about the ocean
Subjective: No
###
Review: This book had a lot of flaws
Subjective: Yes
###
Review: The book was not amazing
Subjective:
------
RESPONSE
------
Yes


In [ ]:

test_prompt_cohere('\n###\n'.join(examples))  # ### is a common few-shot separator

PROMPT:
------
Review: This movie sucks
Subjective: Yes
###
Review: This tv show was about the ocean
Subjective: No
###
Review: This book had a lot of flaws
Subjective: Yes
###
Review: The book was not amazing
Subjective:
------
RESPONSE
------
 Yes


In [ ]:
# Without the examples:
test_prompt_openai('Review: The book was about WWII\nSubjective:')

PROMPT:
------
Review: The book was about WWII
Subjective:
------
RESPONSE
------
Overall, I thought the book was informative and well-written. The author did a good job of providing historical context and presenting the events of World War II in a clear and concise way. I appreciated the depth of research that went into the book and the inclusion of personal stories and anecdotes from individuals who lived through the war. It helped to humanize the historical events and make them more relatable. I also liked how the book examined the various causes and consequences of the war, not just focusing on the military aspects. This added a broader perspective and helped me understand the complexities of the conflict. The only criticism I have is that at times the book delved into too much detail, particularly when discussing specific battles or military strategies. While I understand the importance of these details for a comprehensive understanding of the war, it sometimes made the book feel 

In [ ]:
# With a prompt
test_prompt_openai('Tell me the subjectivity of this review.\n\nReview: The book was about WWII\nSubjective:')

PROMPT:
------
Tell me the subjectivity of this review.

Review: The book was about WWII
Subjective:
------
RESPONSE
------
The book provided a comprehensive and captivating account of the events and impact of WWII, showcasing the bravery and resilience of individuals who lived through the war.


In [ ]:
# Be more specific about the output
test_prompt_openai('Tell me the subjectivity of this review with either "Yes" or "No".\n\nReview: The book was about WWII\nSubjective:')

PROMPT:
------
Tell me the subjectivity of this review with either "Yes" or "No".

Review: The book was about WWII
Subjective:
------
RESPONSE
------
No


In [ ]:
# A different review
test_prompt_openai('Tell me the subjectivity of this review with either "Yes" or "No".\n\nReview: The fight scenes were the best part!\nSubjective:')

PROMPT:
------
Tell me the subjectivity of this review with either "Yes" or "No".

Review: The fight scenes were the best part!
Subjective:
------
RESPONSE
------
Yes


In [ ]:
# Be more specific about the output
test_prompt_openai('Tell me the subjectivity of this review with either "Yes" or "No". Also as a JSON.\n\nReview: The book was about WWII\nSubjective:')


PROMPT:
------
Tell me the subjectivity of this review with either "Yes" or "No". Also as a JSON.

Review: The book was about WWII
Subjective:
------
RESPONSE
------
No

{
  "review": "The book was about WWII",
  "subjective": "No"
}


# Personas / Style

In [ ]:
# It only takes a few words to pretty drastically change the output

In [ ]:
style = 'rude'
rude_response = test_prompt_openai(f'Respond to the customer as a {style} customer service agent.\n\nCustomer: Hey! I cannot seem to get into my account. Can you help?\nAgent:')


PROMPT:
------
Respond to the customer as a rude customer service agent.

Customer: Hey! I cannot seem to get into my account. Can you help?
Agent:
------
RESPONSE
------
What's the problem now? Did you forget your password for the umpteenth time or are you just incapable of following simple instructions?


In [ ]:
style = 'friendly'
friendly_response = test_prompt_openai(f'Respond to the customer as a {style} customer service agent.\n\nCustomer: Hey! I cannot seem to get into my account. Can you help?\nAgent:')


PROMPT:
------
Respond to the customer as a friendly customer service agent.

Customer: Hey! I cannot seem to get into my account. Can you help?
Agent:
------
RESPONSE
------
Hello there! Of course, I'd be happy to help you out with your account. Can you please provide me with your username or email associated with your account so that I can assist you further?


In [ ]:
style = 'pirate'
pirate_response = test_prompt_openai(f'Respond to the customer as a {style} customer service agent.\n\nCustomer: Hey! I cannot seem to get into my account. Can you help?\nAgent:')


PROMPT:
------
Respond to the customer as a pirate customer service agent.

Customer: Hey! I cannot seem to get into my account. Can you help?
Agent:
------
RESPONSE
------
Arrr, me heartie! Ye be havin' trouble with yer account, eh? Fear not, for I be here to lend ye a hand on this treacherous sea of technology! Tell me, have ye tried walkin' the plank, I mean, resetting yer password?


In [ ]:
style = 'yoda'
test_prompt_openai(f'Respond to the customer as a {style} customer service agent.\n\nCustomer: Hey! I cannot seem to get into my account. Can you help?\nAgent:')


PROMPT:
------
Respond to the customer as a yoda customer service agent.

Customer: Hey! I cannot seem to get into my account. Can you help?
Agent:
------
RESPONSE
------
Help you, I can. Account access, I will restore. Provide me with your email and password, you must.


# Output validation and bias!

In [ ]:
!pip install transformers

In [ ]:
from transformers import pipeline

# Initialize the zero-shot-classification pipeline using the BART-MNLI model
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Define candidate labels for classification
candidate_labels = ['offensive', 'safe']

# Classify the responses using the classifier and allow for multiple labels
# This will return a dictionary with the sequence, labels, and their corresponding scores
rude_response = "What's the problem now? Did you forget your password for the umpteenth time or are you just incapable of following simple instructions?"
# Classify the rude response using the classifier
classifier(rude_response, candidate_labels, multi_label=True)



{'sequence': "What's the problem now? Did you forget your password for the umpteenth time or are you just incapable of following simple instructions?",
 'labels': ['offensive', 'safe'],
 'scores': [0.0392235703766346, 0.00030594199779443443]}

In [ ]:
friendly_response = "Hello there! Of course, I'd be happy to help you out with your account. Can you please provide me with your username or email associated with your account so that I can assist you further?"
classifier(friendly_response, candidate_labels, multi_label=True)

{'sequence': "Hello there! Of course, I'd be happy to help you out with your account. Can you please provide me with your username or email associated with your account so that I can assist you further?",
 'labels': ['safe', 'offensive'],
 'scores': [0.12946759164333344, 0.017940092831850052]}

In [ ]:
# Pirate
pirate_response = "Arrr, me heartie! Ye be havin' trouble with yer account, eh? Fear not, for I be here to lend ye a hand on this treacherous sea of technology! Tell me, have ye tried walkin' the plank, I mean, resetting yer password?"
classifier(pirate_response, candidate_labels, multi_label=True)


{'sequence': "Arrr, me heartie! Ye be havin' trouble with yer account, eh? Fear not, for I be here to lend ye a hand on this treacherous sea of technology! Tell me, have ye tried walkin' the plank, I mean, resetting yer password?",
 'labels': ['offensive', 'safe'],
 'scores': [0.04164835438132286, 0.02613351307809353]}

**temperature** : regulates the unpredictability of a language model's output. With higher temperature settings, outputs become more creative and less predictable as it amplifies the likelihood of less probable tokens while reducing that for more probable ones. Conversely, lower temperatures yield more conservative and predictable results.

In [9]:
#changing the temperature
from tqdm import tqdm

style = 'friendly'
responses = []
for _ in tqdm(range(10)):
    responses.append(test_prompt_openai(
        f'Respond to the customer as a {style} customer service agent.\n\nCustomer: Hey! I cannot seem to get into my account. Can you help?\nAgent:',
        temperature=0,
        suppress=True
    ))


100%|██████████| 10/10 [00:22<00:00,  2.22s/it]


In [10]:
# only 1 unique response
responses, len(set(responses))

(["Hi there! I'd be happy to help you with that. Could you please provide me with your account username or email address so that I can assist you further?",
  "Hi there! I'd be happy to help you with that. Could you please provide me with your account username or email address so that I can assist you further?",
  "Hi there! I'd be happy to help you with that. Could you please provide me with your account username or email address so that I can assist you further?",
  "Hi there! I'd be happy to help you with that. Could you please provide me with your account username or email address so that I can assist you further?",
  "Hi there! I'd be happy to help you with that. Could you please provide me with your account username or email address so that I can assist you further?",
  "Hi there! I'd be happy to help you with that. Could you please provide me with your account username or email address so that I can assist you further?",
  "Hi there! I'd be happy to help you with that. Could you

In [ ]:
style = 'friendly'
responses = []
for _ in tqdm(range(10)):
    responses.append(test_prompt_openai(
        f'Respond to the customer as a {style} customer service agent.\n\nCustomer: Hey! I cannot seem to get into my account. Can you help?\nAgent:',
        temperature=0.9,
        suppress=True
    ))


100%|██████████| 10/10 [00:27<00:00,  2.79s/it]


In [ ]:
responses, len(set(responses))

(["Hello! I'm sorry to hear that you're having trouble accessing your account. I'd be happy to assist you with that. Could you please provide me with your account username or email address so that I can look into it for you?",
  "Hey there! Of course, I'd be happy to help you get back into your account. Can you please provide me with some more details about the issue you're experiencing?",
  "Hi there! I'd be happy to help you with your account. Could you please provide me with your username or email address so that I can assist you further?",
  "Hi there! Of course, I'd be happy to help you regain access to your account. Can you please provide me with your username or email associated with the account?",
  "Hello! I'd be happy to assist you with accessing your account. Could you please provide me with your username or email address?",
  "Hi there! I'm sorry to hear that you're having trouble accessing your account. I'd be happy to help you out. Could you please provide me with your ac

**Top_p**: helps manage the randomness of the model's output. It works by establishing a probability threshold and then selecting tokens whose combined likelihood surpasses this limit.

In [ ]:
from tqdm import tqdm

style = 'friendly'
responses = []
for _ in tqdm(range(10)):
    responses.append(test_prompt_openai(
        f'Respond to the customer as a {style} customer service agent.\n\nCustomer: Hey! I cannot seem to get into my account. Can you help?\nAgent:',
        temperature= 0.9,
        top_p=.1,

        suppress=True
    ))
# restricting top p allows fewer tokens to be considered, making the model more deterministic
responses, len(set(responses))


100%|██████████| 10/10 [00:24<00:00,  2.48s/it]


(["Hi there! I'd be happy to help you with that. Could you please provide me with your account username or email address so that I can assist you further?",
  "Hi there! I'd be happy to help you with that. Could you please provide me with your account username or email address so that I can assist you further?",
  "Hi there! I'd be happy to help you with that. Could you please provide me with your account username or email address so that I can assist you further?",
  "Hi there! I'd be happy to help you with that. Could you please provide me with your account username or email address so that I can assist you further?",
  "Hi there! I'd be happy to help you with that. Could you please provide me with your account username or email address so that I can assist you further?",
  "Hi there! I'd be happy to help you with that. Could you please provide me with your account username or email address so that I can assist you further?",
  "Hi there! I'd be happy to help you with that. Could you

Pinecone is a vector database that is designed for small to medium-sized datasets. Pinecone is optimised for fast vector search and retrieval.

 The function get_best_result_from_pinecone () makes a POST request to a Pinecone-powered information retrieval service to retrieve documents based on a specified query.

In [11]:
NAMESPACE = 'default'  # pinecone namespace

In [12]:
import requests
import json

def get_best_result_from_pinecone(query, namespace=NAMESPACE):
    payload = json.dumps({
      "num_results": 2,
      "query": query,
      "re_ranking_strategy": "none",
      "namespace": namespace
    })

    response = requests.post(
        "https://information-retrieval-hiaa.onrender.com/document/retrieve",
        data=payload
    )

    return response.json()['documents'][0]


In [17]:
get_best_result_from_pinecone("What is a Mideterenian Diet?")  #Notice the spelling of Mediterranean!

{'text': 'Italian meal structure is typical of the Mediterranean region and different from meal structure of Northern Europe / Northwestern Europe and Germanic and Slavic Europe, though it still often consists of breakfast, lunch, and supper. However, much less emphasis is placed on breakfast, and breakfast itself is often skipped or involves lighter meal portions than are seen in other non-Mediterranean Western countries. Late-morning and mid-afternoon snacks, called merenda (plural merende), are also often included in this meal structure. Italians also commonly divide a celebratory meal into several different courses.',
 'date_uploaded': '2023-08-14T14:59:43.370141',
 'score': 0.829652548,
 'id': 'ebff91140f453bbb247b29ca685e6011'}

In [18]:
get_best_result_from_pinecone("How old is Obama?")

{'text': "In November 2008, the show's post-election day telecast garnered the biggest audience in the show's history at 6.2 million in total viewers, becoming the week's most-watched program in daytime television. It was surpassed on July 29, 2010, during which former President Barack Obama first appeared as a guest on The View, which garnered a total of 6.6 million viewers. In 2013, the show was reported to be averaging 3.1 million daily viewers, which outpaced rival talk show The Talk.",
 'date_uploaded': '2023-08-14T15:00:48.569014',
 'score': 0.788054585,
 'id': 'be0b9cb01bae3d7fa13839b85c10079c'}

In [19]:
query = "How old is Obama?"

best_result = get_best_result_from_pinecone(query)

PROMPT = f"""
Answer the question using the context.

Context: {best_result['text']}
Query: {query}
Answer:""".strip()

test_prompt_openai(PROMPT)

PROMPT:
------
Answer the question using the context.

Context: In November 2008, the show's post-election day telecast garnered the biggest audience in the show's history at 6.2 million in total viewers, becoming the week's most-watched program in daytime television. It was surpassed on July 29, 2010, during which former President Barack Obama first appeared as a guest on The View, which garnered a total of 6.6 million viewers. In 2013, the show was reported to be averaging 3.1 million daily viewers, which outpaced rival talk show The Talk.
Query: How old is Obama?
Answer:
------
RESPONSE
------
The information provided in the context does not mention the age of Barack Obama.


In [20]:
# Asking it to not answer if the context isn't enough.
query = "How old is Obama?"

best_result = get_best_result_from_pinecone(query)

PROMPT = f"""
Only using the following context, answer the question. If you cannot answer using the context, say 'I don't know. Use this format:

Context: (context)
Query: (natural language query)
Answer: (answer)

Context: {best_result['text']}
Query: {query}
Answer:""".strip()

test_prompt_openai(PROMPT)


PROMPT:
------
Only using the following context, answer the question. If you cannot answer using the context, say 'I don't know. Use this format:

Context: (context)
Query: (natural language query)
Answer: (answer)

Context: In November 2008, the show's post-election day telecast garnered the biggest audience in the show's history at 6.2 million in total viewers, becoming the week's most-watched program in daytime television. It was surpassed on July 29, 2010, during which former President Barack Obama first appeared as a guest on The View, which garnered a total of 6.6 million viewers. In 2013, the show was reported to be averaging 3.1 million daily viewers, which outpaced rival talk show The Talk.
Query: How old is Obama?
Answer:
------
RESPONSE
------
I don't know.


In [21]:
# use some chain of thought prompting to make it think through the reasoning.
def gen_Q_A(query, qa_engine='openai'):
    best_result = get_best_result_from_pinecone(query)

    PROMPT = f"""
Only using the following context, answer the question.

Context: (context)
Query: (natural language query)
Answer: (answer)
Reasoning: (the reasoning as to how the query is answerable or not using the context)

Context: {best_result['text']}
Query: {query}
Answer:""".strip()

    if qa_engine == 'openai':
        return test_prompt_openai(PROMPT)
    elif qa_engine == 'cohere':
        return test_prompt_cohere(PROMPT)


In [22]:
gen_Q_A('How old is Obama?', qa_engine='openai')

PROMPT:
------
Only using the following context, answer the question.

Context: (context)
Query: (natural language query)
Answer: (answer)
Reasoning: (the reasoning as to how the query is answerable or not using the context)

Context: In November 2008, the show's post-election day telecast garnered the biggest audience in the show's history at 6.2 million in total viewers, becoming the week's most-watched program in daytime television. It was surpassed on July 29, 2010, during which former President Barack Obama first appeared as a guest on The View, which garnered a total of 6.6 million viewers. In 2013, the show was reported to be averaging 3.1 million daily viewers, which outpaced rival talk show The Talk.
Query: How old is Obama?
Answer:
------
RESPONSE
------
(answer not possible with given context)
Reasoning: The given context does not provide any information about the age of Barack Obama.


In [23]:
gen_Q_A('How old is Obama?', qa_engine='cohere')

PROMPT:
------
Only using the following context, answer the question.

Context: (context)
Query: (natural language query)
Answer: (answer)
Reasoning: (the reasoning as to how the query is answerable or not using the context)

Context: In November 2008, the show's post-election day telecast garnered the biggest audience in the show's history at 6.2 million in total viewers, becoming the week's most-watched program in daytime television. It was surpassed on July 29, 2010, during which former President Barack Obama first appeared as a guest on The View, which garnered a total of 6.6 million viewers. In 2013, the show was reported to be averaging 3.1 million daily viewers, which outpaced rival talk show The Talk.
Query: How old is Obama?
Answer:
------
RESPONSE
------
 43 as of November 2008, 46 as of July 2010, or 51 as of 2013 (assuming he was born in 1961)
Reasoning: The context paragraph mentions Obama's age in relation to The View's viewership in various years and does not provide an 

In [26]:
# open source LLMs

from transformers import AutoTokenizer, T5ForConditionalGeneration

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

In [25]:
PROMPT = f"Context: {best_result['text']}\n\nQuestion: {query}\n\nAnswer:"

print(PROMPT)

Context: In November 2008, the show's post-election day telecast garnered the biggest audience in the show's history at 6.2 million in total viewers, becoming the week's most-watched program in daytime television. It was surpassed on July 29, 2010, during which former President Barack Obama first appeared as a guest on The View, which garnered a total of 6.6 million viewers. In 2013, the show was reported to be averaging 3.1 million daily viewers, which outpaced rival talk show The Talk.

Question: How old is Obama?

Answer:


In [27]:
encoded_prompt = tokenizer(PROMPT, return_tensors="pt")
output = model.generate(**encoded_prompt, max_new_tokens=256)
tokenizer.decode(output[0], skip_special_tokens=True)

'former'

In [30]:
PROMPT = f"Context: {best_result['text']}\n\nQuestion: How many people watched the telecast?\n\nAnswer:"

encoded_prompt = tokenizer(PROMPT, return_tensors="pt")
output = model.generate(**encoded_prompt, max_new_tokens=256)
tokenizer.decode(output[0], skip_special_tokens=True)

'6.2 million'

# Prompt Chaining

In [ ]:
email = 'Hey John,\n\nI will not lie, I am a bit upset about the speed at which my organization is moving but I wanted to ask if you were still interested in working with us.\n\nBest,\nCharles'

print(email)

Hey John,

I will not lie, I am a bit upset about the speed at which my organization is moving but I wanted to ask if you were still interested in working with us.

Best,
Charles


In [ ]:
# not the most empathetic reply
test_prompt_openai(f'Write an email back.\n\nEmail: {email}\n\nResponse:')

PROMPT:
------
Write an email back.

Email: Hey John,

I will not lie, I am a bit upset about the speed at which my organization is moving but I wanted to ask if you were still interested in working with us.

Best,
Charles

Response:
------
RESPONSE
------
Hi Charles,

I understand your frustration with the speed of progress in our organization. I apologize if it has caused any inconvenience or disappointment. Rest assured that I am committed to working with you and ensuring that we address any concerns or issues that may be hindering our progress.

If there are specific areas or projects that you would like to discuss or prioritize, please let me know. It's important for us to have open communication and align our efforts towards a common goal. I value your input and want to make sure we are moving in the right direction together.

Thank you for reaching out and expressing your concerns. Let's schedule a meeting to discuss this further and find ways to improve our efficiency and speed

In [ ]:
test_prompt_openai(f'Write an email back taking their feelings into consideration.\n\nEmail: {email}\n\nResponse:')

PROMPT:
------
Write an email back taking their feelings into consideration.

Email: Hey John,

I will not lie, I am a bit upset about the speed at which my organization is moving but I wanted to ask if you were still interested in working with us.

Best,
Charles

Response:
------
RESPONSE
------
Hi Charles,

I appreciate your candidness and I understand your frustration with the pace of our organization. I want you to know that I value your input and concerns.

I assure you that your interest in working with us is still of great importance to us. We believe that your skills and expertise would be valuable assets to our team. Despite the setbacks we have faced, we are determined to address the ongoing challenges and strive for improvement.

Please know that we are diligently working towards making necessary changes to overcome any obstacles and accelerate our progress. Your patience and understanding during this process would be highly appreciated.

If you have any specific suggestions

# Building Chatbots using RAG (Retrieval Augmented Generation)

In [31]:
SYSTEM_PROMPT = '''
You are a helpful Q/A bot that can only use reference material from a knowledge base.
If a user asks anything that is not "from the knowledge base", say that you cannot answer.
'''.strip()

class ChatbotGPT():
    def __init__(self, system_prompt, threshold=.8):
        self.conversation = [{'role': 'system', 'content': system_prompt}]
        self.threshold = threshold

    def display_conversation(self):
        '''display the conversation in a pretty format denoting the system, user and assistant differently'''
        for turn in self.conversation:
            role = turn['role']
            content = turn['content']
            if role == 'system':
                print(f'System: {content}')
            elif role == 'user':
                print(f'User: {content}')
            elif role == 'assistant':
                print(f'Assistant: {content}')
            print('------------')

    def user_turn(self, message):
        self.conversation.append({"role": "user", "content": message})
        best_result = get_best_result_from_pinecone(message)
        print(best_result)
        if best_result['score'] >= self.threshold:
            # Add to the context to the system prompt
            self.conversation[0]['content']+=f'\n\nFrom the knowledge base: "{best_result["text"]}"'

        chatgpt_response = openai_client.chat.completions.create(
            model='gpt-3.5-turbo',
            temperature=0,
            messages=self.conversation
        ).choices[0].message.content.strip()
        self.conversation.append({'role': 'assistant', 'content': chatgpt_response})
        return self.conversation[-1]


In [33]:
c = ChatbotGPT(system_prompt=SYSTEM_PROMPT)

c.user_turn('what are fixed costs?')

{'text': 'In economics, fixed costs, indirect costs or overheads are business expenses that are not dependent on the level of goods or services produced by the business. They tend to be time-related, such as salaries or rents being paid per month, and are often referred to as overhead costs. This is in contrast to variable costs, which are volume-related (and are paid per quantity produced). For a simple example, such as a bakery, the monthly rent for the baking facilities, and the monthly payments for the security system and basic phone line are fixed costs, as they do not change according to how much bread the bakery produces and sells. On the other hands, the wage costs of the bakery are variable, as the bakery will have to hire more workers if the production of bread increases. The relation between fixed cost and variable cost can be modelled by an analytical formula.', 'date_uploaded': '2023-08-14T15:04:37.247682', 'score': 0.889511824, 'id': '57a0103e5716168be7498b4531b21d07'}


{'role': 'assistant',
 'content': 'Fixed costs, also known as indirect costs or overheads, are business expenses that do not vary with the level of goods or services produced by the business. These costs are typically time-related and remain constant regardless of the quantity produced. Examples of fixed costs include rent, salaries, insurance, utilities, and equipment leases. Unlike variable costs, which change with production levels, fixed costs remain the same over a given period.'}

In [34]:
c.user_turn('How old is Obama?')

{'text': "In November 2008, the show's post-election day telecast garnered the biggest audience in the show's history at 6.2 million in total viewers, becoming the week's most-watched program in daytime television. It was surpassed on July 29, 2010, during which former President Barack Obama first appeared as a guest on The View, which garnered a total of 6.6 million viewers. In 2013, the show was reported to be averaging 3.1 million daily viewers, which outpaced rival talk show The Talk.", 'date_uploaded': '2023-08-14T15:00:48.569014', 'score': 0.788054585, 'id': 'be0b9cb01bae3d7fa13839b85c10079c'}


{'role': 'assistant',
 'content': 'I\'m sorry, but I cannot answer that question as it is not "from the knowledge base."'}

In [35]:
c.user_turn('thanks so much!')

{'text': "Susie moves on into another, larger part of heaven, occasionally watching earthbound events. Lindsey and Samuel have a daughter together named Abigail Suzanne. While stalking a young woman in New Hampshire, Harvey is hit on the shoulder by an icicle and falls to his death down a snow-covered slope into the ravine below. At the end of the novel, a Norristown couple finds Susie's charm bracelet but don't realize its significance, and Susie closes the story by wishing the reader ``a long and happy life''.", 'date_uploaded': '2023-08-14T15:03:33.926641', 'score': 0.735897, 'id': '8ae1783aed03559f78a886c77e8001ff'}


{'role': 'assistant',
 'content': "You're welcome! If you have any more questions related to the knowledge base, feel free to ask."}

In [36]:
c.display_conversation()

System: You are a helpful Q/A bot that can only use reference material from a knowledge base.
If a user asks anything that is not "from the knowledge base", say that you cannot answer.

From the knowledge base: "In economics, fixed costs, indirect costs or overheads are business expenses that are not dependent on the level of goods or services produced by the business. They tend to be time-related, such as salaries or rents being paid per month, and are often referred to as overhead costs. This is in contrast to variable costs, which are volume-related (and are paid per quantity produced). For a simple example, such as a bakery, the monthly rent for the baking facilities, and the monthly payments for the security system and basic phone line are fixed costs, as they do not change according to how much bread the bakery produces and sells. On the other hands, the wage costs of the bakery are variable, as the bakery will have to hire more workers if the production of bread increases. The r

# Dynamic k shot using embeddings WITH GSM8K

In [ ]:
!pip install datasets


In [39]:
from datasets import load_dataset
gsm_dataset = load_dataset("gsm8k", "main")
print(json.dumps(gsm_dataset['train'][0], indent=4))

{
    "question": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
    "answer": "Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72"
}


In [40]:
gsm_dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

In [41]:
print(gsm_dataset['test']['question'][0])
print()
print(gsm_dataset['test']['answer'][0])

Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


In [42]:
#Define a function to format k-shot examples for GSM
def format_k_shot_gsm(examples, cot=True):
    if cot:
        #If cot=True, include the reasoning in the prompt
        return '\n###\n'.join(
            [f'Question: {e["question"]}\nReasoning: {e["answer"].split("####")[0].strip()}\nAnswer: {e["answer"].split("#### ")[-1]}' for e in examples]
        )
    else:
        #If cot=False, exclude the reasoning in the prompt
        return '\n###\n'.join(
            [f'Question: {e["question"]}\nAnswer: {e["answer"].split("#### ")[-1]}' for e in examples]
        )


In [43]:
unanswered_example = gsm_dataset['test'][4]
print(unanswered_example)

PROMPT = f"""Answer the arithmetic problem in the following format:

{format_k_shot_gsm(list(gsm_dataset['train'])[:3])}
###
Question: {unanswered_example["question"]}
Reasoning:""".strip()

print(PROMPT)

{'question': "Every day, Wendi feeds each of her chickens three cups of mixed chicken feed, containing seeds, mealworms and vegetables to help keep them healthy.  She gives the chickens their feed in three separate meals. In the morning, she gives her flock of chickens 15 cups of feed.  In the afternoon, she gives her chickens another 25 cups of feed.  How many cups of feed does she need to give her chickens in the final meal of the day if the size of Wendi's flock is 20 chickens?", 'answer': 'If each chicken eats 3 cups of feed per day, then for 20 chickens they would need 3*20=<<3*20=60>>60 cups of feed per day.\nIf she feeds the flock 15 cups of feed in the morning, and 25 cups in the afternoon, then the final meal would require 60-15-25=<<60-15-25=20>>20 cups of chicken feed.\n#### 20'}
Answer the arithmetic problem in the following format:

Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altog

In [44]:
test_prompt_openai(PROMPT)

PROMPT:
------
Answer the arithmetic problem in the following format:

Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Reasoning: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
Answer: 72
###
Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Reasoning: Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
Answer: 10
###
Question: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Reasoning: In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandpare

In [45]:
#The correct answer
print(gsm_dataset['test']['answer'][4])

If each chicken eats 3 cups of feed per day, then for 20 chickens they would need 3*20=<<3*20=60>>60 cups of feed per day.
If she feeds the flock 15 cups of feed in the morning, and 25 cups in the afternoon, then the final meal would require 60-15-25=<<60-15-25=20>>20 cups of chicken feed.
#### 20


In [46]:
test_prompt_cohere(PROMPT)

PROMPT:
------
Answer the arithmetic problem in the following format:

Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Reasoning: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
Answer: 72
###
Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Reasoning: Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
Answer: 10
###
Question: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Reasoning: In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandpare